# 10 — Volatility overlay: does braking in volatile months help?

Notebook 09 found that the 252-day volatility forecast is far too low in a few
stress months. A volatility overlay reacts to exactly those months: it moves
part of a run into T-bills when the run's own recent volatility is high, and
never adds risk. This notebook applies one such overlay, the volatility-managed
portfolio (VMP), to the eight core runs of notebook 08 and to 60/40. Two
questions are tested, as registered in §1: does the overlay change each run's
Sharpe ratio (H1), and does any overlaid run differ from 60/40 (H2)?
Performance, crisis episodes and sensitivities are descriptive. H1 and H2 are
this notebook's own (not notebook 08's).

## Setup

Data and run labels as in notebook 08; `VMP(m)` is the overlay on run m.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from helpers import nb10 as h
from maplab import Panel, apply_style

In [ ]:
apply_style()

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

## §1 What we test

Registered before any overlay return, Sharpe ratio, volatility or drawdown was
computed (a read-only pre-check had printed only exposure, turnover and cost
drag); the frozen text is in `registrations/nb10.toml`. Tests, Newey–West lags
and the bootstrap are notebook 08's.

| | Question | Test | Rejected if |
|---|---|---|---|
| H1 (8 tests) | Does the overlay change each core run's Sharpe ratio, full window? | Sharpe-difference test with Newey–West errors; Romano–Wolf over the 8 | Romano–Wolf p < 0.05 and the test-window difference has the same sign |
| H2 (8 tests) | Does each overlaid core run's Sharpe ratio differ from 60/40's? | Same test and adjustment | Same rule |

An adjusted p between 0.05 and 0.10 holds "narrowly"; Holm p is reported as a
description. VMP(60/40) − 60/40, performance, exposure, the 2020 and 2022
episodes and four sensitivities (63-day window, inverse variance, cap 1.5,
month-end updates) are descriptive and never tested. The overlay's parameters
were fixed in advance, not tuned.

Limits: the base results and an earlier, flawed version of this overlay (full-sample volatility target, free leverage, no overlay costs) were seen before this
design; the tests are asymptotic and have limited power, although VMP(m) and m
are highly correlated, which narrows the standard error of their difference.

In [ ]:
reg = ml.inference.load_registration("nb10")
adj = [x["id"] for x in reg["hypothesis"] if x["kind"] == "adjusted_null"]
assert adj == h.H1_IDS + h.H2_IDS

## §2 The overlay and the statistics

General form of a volatility-managed portfolio: scale a portfolio's return by the inverse of its recent variance, $r^{VM}_{t} = (k/\hat\sigma^2_{t-1})\,r_t$, with $k$ set so the managed and unmanaged series have the same unconditional volatility. That needs leverage and a full-sample constant.

Our version, registered in §1, uses neither. In words: each day the overlay compares the run's volatility over the last month with the average of its own past readings; when recent volatility is higher, it holds only the fraction $c_t$ of the run and puts the rest in T-bills (BIL). For run $m$ with daily net return $r_t$ and BIL return $f_t$:

$$\hat\sigma_t = \sqrt{252}\;\mathrm{sd}\big(r_s - f_s,\ s = t-20,\dots,t\big), \qquad \sigma^*_t = \mathrm{mean}\big(\hat\sigma_s,\ s \le t-1\big),$$

$$c_t = \min\big(1,\ \sigma^*_t / \hat\sigma_{t-1}\big), \qquad c_t = 1 \text{ for the first 252 days},$$

$$r^{VMP}_t = c_t\,r_t + (1-c_t)\,f_t - \kappa\,\lvert c_t - d_{t-1}\rvert, \qquad d_t = \frac{c_t\,(1+r_t)}{1 + c_t r_t + (1-c_t) f_t}, \qquad \kappa = 10\ \text{bp}.$$

$d_{t-1}$ is yesterday's risky fraction after drift, so the cost is charged on the exact trade. With $c \le 1$ the overlay only brakes: it moves part of the book into T-bills when trailing volatility is above its own history and never adds risk when it is below. The base method's trading costs are already in $r_t$; the overlay adds only the risky-to-BIL trade.

Tests: the delta-method Newey–West Sharpe difference, stationary bootstrap and Romano–Wolf step-down of notebook 08 §2, applied to $\Delta SR = SR(\mathrm{VMP}(m)) - SR(m)$ (H1) and $SR(\mathrm{VMP}(m)) - SR(60/40)$ (H2) on daily excess returns from 2010-02-02. Romano–Wolf uses $t^* = (\Delta SR^* - \Delta\widehat{SR})/\mathrm{se}_{NW}$, centred on the bootstrap's own estimator (ddof 1).

## §3 Runs and overlays

The eight core runs and 60/40, rebuilt with notebook 08's constructors; their Sharpe ratios match notebook 08's (`tests/test_reproduction.py` checks this). Everything below is scored from 2010-02-02, the first day every overlay is live after its 252-day warm-up, so the base runs' Sharpe ratios here differ from notebook 08's, which start in February 2009.

In [ ]:
runs = h.build_base_runs(simple_returns, panel)

Overlays with the registered parameters. `tests/test_nb10_checks.py` checks that with $c \equiv 1$ each overlay returns its base series exactly, that the net return rebuilt from $h$, $r$, BIL and the cost matches the overlay's, and that every overlay goes live on 2010-02-02.

In [ ]:
ov = h.build_overlays(runs, rf_daily)
all_runs = {**runs, **ov}
assert list(ov) == h.VMPS

No look-ahead: a +5% shock to the base return on 2020-03-16 leaves $c$ unchanged through that day and moves it the next day.

In [ ]:
dc_next = {}
for name in ("EW", "60/40"):
    sc = h.shock_check(runs, rf_daily, name, "2020-03-16")
    assert sc["max_abs_dc_through_at"] == 0.0 and sc["dc_next_day"] != 0.0
    dc_next[name] = sc["dc_next_day"]
print(f"look-ahead test passed: a +5% shock on 2020-03-16 leaves c unchanged through that day "
      f"and moves it the next (EW {dc_next['EW']:+.4f}, 60/40 {dc_next['60/40']:+.4f})")

Scoring windows start on 2010-02-02 (4,086 days; 834 in the test window).

In [ ]:
XW = h.scoring_frames(all_runs, h.BASES + h.VMPS, rf_daily, split_ts)
assert [len(Xw) for Xw in XW.values()] == [4086, 3252, 834]
assert [ml.robust.nw_lag(len(Xw)) for Xw in XW.values()] == [9, 8, 6]
assert XW["full"].index[0] == h.LIVE

### 3.1 Exposure

Target fraction $c$, held fraction $h$ (equal under daily updates), share of days braking ($h < 1$), the lowest $c$ and its date, and the overlay's own turnover and cost drag, full window from 2010-02-02.

In [ ]:
ex = h.exposure_table(ov, split_ts)
exf = ex.xs("full", level="window")

exp_full = pd.DataFrame({
    "mean c": exf["mean_c"], "share braking (%)": exf["share_braking"], "lowest c": exf["min_c"],
    "date": exf["min_c_date"].dt.strftime("%Y-%m-%d"),
    "overlay turnover (%/yr)": exf["ann_overlay_turnover"], "cost drag (bp/yr)": exf["cost_drag_bps"],
}).rename_axis(None)
display(ml.reporting.show_table(exp_full, pct=["share braking (%)", "overlay turnover (%/yr)"],
                                int_cols=["share braking (%)", "overlay turnover (%/yr)", "cost drag (bp/yr)"]))

**Reading.** The overlay brakes on 28–42% of days and is fully invested otherwise. Mean $c$ is 0.87–0.94, lowest for MaxSharpe(S). Every lowest $c$ falls between 2020-03-25 and 2020-04-07, at 0.14–0.21, and cost drag is 19–28 bp/yr, highest for MaxSharpe(S).

Held fraction $h$ over time: median and range across the eight core overlays, with VMP(60/40); the table gives the median core overlay's mean $h$ per calendar year.

In [ ]:
fig = h.exposure_figure(ov, h.CORE)
plt.show()
assert len(fig.axes) == 1 and len(fig.axes[0].get_lines()) == 2

ml.reporting.show_table(h.exposure_by_year(ov, h.CORE)[["core median"]], dp=2)

**Reading.** The overlay holds least in the stress years: the median core overlay holds least in 2022 (0.77) and 2020 (0.85) among full calendar years; 2026 to date (January–April) is 0.84. Its lowest single day is 2020-04-07 (median $h$ 0.17).

## §4 Performance

Base runs, their overlays, 60/40 and VMP(60/40), from 2010-02-02: Sharpe ratio on the full and test windows, everything else on the full window. For a VMP row, turnover and cost drag are the overlay's own risky-to-BIL trading; the base method's trading is already in its net return.

In [ ]:
ORDER = [x for m in h.BASES for x in (m, h.vmp(m))]
perf = h.perf_table(all_runs, ORDER, split_ts, rf_daily)
perf = perf.reindex(pd.MultiIndex.from_product([ORDER, list(h.WINDOWS)], names=["strategy", "window"]))

pf = perf.xs("full", level="window")
perf_shown = pd.DataFrame({
    "Sharpe full": pf["sharpe"],
    "Sharpe test": perf.xs("test (> split)", level="window")["sharpe"],
    "return (%)": pf["ann_return"], "vol (%)": pf["ann_vol"], "max DD (%)": pf["max_dd"],
    "Calmar": pf["calmar"], "turnover (%/yr)": pf["ann_turnover"], "cost drag (bp/yr)": pf["cost_drag"] * 1e4,
}).loc[ORDER].rename_axis(None)
ml.reporting.show_table(perf_shown, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"], pct_dp=1,
                        int_cols=["turnover (%/yr)", "cost drag (bp/yr)"])

**Reading.** The overlay trades return for lower volatility, and Sharpe barely moves. In the full window it lowers both volatility and return for all nine runs; volatility falls to 73% (MaxSharpe(S)) to 87% (MDP(S)) of the base. Sharpe rises for four runs (60/40 +0.09, BMV(0.3) +0.05, MaxSharpe(S) +0.02, HRP(S) +0.01) and falls for five, most for MDP(S) (−0.08) and GMV(S) (−0.07). Maximum drawdown is shallower for seven; GMV(S) and MDP(S) are slightly deeper. VMP(60/40) has the highest full-window Sharpe of the 18 rows (0.96), but in the test window it is below 60/40 (0.85 against 1.00).

Each arrow runs from a base run (hollow) to its overlay (filled) in (volatility, return), full window from 2010-02-02.

In [ ]:
fig = h.arrows_figure(perf, h.BASES)
plt.show()
assert len(fig.axes) == 1

**Reading.** The overlay helps Sharpe where it cuts volatility most. The three largest volatility cuts, MaxSharpe(S), 60/40 and BMV(0.3), are three of the four runs whose Sharpe rises; the two smallest, MDP(S) and GMV(S), are the two largest Sharpe losses.

## §5 H1 — overlay vs base

Registered test (§1): $\Delta SR = SR(\mathrm{VMP}(m)) - SR(m)$ on the full window, delta-method Newey–West $t$, Romano–Wolf over the 8, with the test-window $\Delta SR$, the bootstrap 95% interval and Holm (descriptive).

In [ ]:
SR_star, IDX_full = h.bootstrap_full(XW, h.BASES + h.VMPS)
assert sum(c.shape[0] for c in IDX_full) == h.B and all(c.shape[1] == len(XW["full"]) for c in IDX_full)

In [ ]:
DSR_NAMES = {"dSR_full": "ΔSR full", "se": "se", "t": "t", "p_raw": "p", "p_holm": "Holm p",
             "p_rw": "Romano–Wolf p", "dSR_test": "ΔSR test", "same_sign_test": "same sign in test",
             "ci_lo": "95% lo", "ci_hi": "95% hi"}


def dsr_shown(tbl, run_col, cols=DSR_NAMES):
    shown = tbl.set_index(run_col)[list(cols)].rename(columns=cols).rename_axis("run")
    p_cols = [c for c in ("p", "Holm p", "Romano–Wolf p", "bootstrap p") if c in shown]
    return ml.reporting.show_table(shown, dp=2, col_dp={c: 3 for c in p_cols})


h1, h1_stats, _ = h.dsr_family(XW, h.H1_PAIRS, h.H1_IDS, SR_star)
assert list(h1.index) == h.H1_IDS and (h1["L"] == 9).all() and (h1["L_test"] == 6).all()
dsr_shown(h1, "b")

**Reading.** No overlay changes its base run's Sharpe ratio detectably. ΔSR runs from −0.08 (MDP(S)) to +0.05 (BMV(0.3)); the largest $|t|$ is 1.23 and the smallest Romano–Wolf p 0.451. The standard errors, 0.07 (MDP(S)) to 0.12 (ERC(S)), are below notebook 08's 0.10 to 0.26 because VMP(m) equals m on most days; an unadjusted 5% rejection would still need a gap of about 0.13 to 0.24, against an observed maximum of 0.08. The test-window ΔSR keeps the full-window sign for 7 of the 8 (not BMV(0.3)).

## §6 H2 — overlay vs 60/40

Same test for $\Delta SR = SR(\mathrm{VMP}(m)) - SR(60/40)$. VMP(60/40) − 60/40 is shown below the family as a descriptive row; it is not tested.

In [ ]:
h2, h2_stats, _ = h.dsr_family(XW, h.H2_PAIRS, h.H2_IDS, SR_star)
assert list(h2.index) == h.H2_IDS and (h2["L"] == 9).all() and (h2["L_test"] == 6).all()
display(dsr_shown(h2, "a"))

d6040, _, _ = h.dsr_family(XW, [h.D_PAIR], ["D_6040"], SR_star)
dsr_shown(d6040.assign(run="VMP(60/40) − 60/40"), "run",
          {k: DSR_NAMES[k] for k in ("dSR_full", "se", "t", "p_raw", "dSR_test", "ci_lo", "ci_hi")})

**Reading.** No overlaid core run beats 60/40 over the full window, and none is detectably worse. Every VMP(m) has a lower full-window Sharpe than 60/40, from −0.35 (BL(0.1,S)) to −0.003 (BMV(0.3)); the largest $|t|$ is 1.93 (BL(0.1,S), Romano–Wolf p 0.198). BL(0.1,S)'s bootstrap 95% interval just excludes 0 (upper end −0.008) while $|t| < 2$: its upper bootstrap tail (0.34) is shorter than 1.96 × se (0.35). In the test window VMP(MaxSharpe(S)) and VMP(BMV(0.3)) are above 60/40. The standard errors, 0.13 to 0.28 against H1's 0.07 to 0.12, leave little power, so these nulls are weak evidence of equal Sharpe ratios. VMP(60/40) − 60/40 (descriptive) is +0.09 (t 0.87) in the full window and −0.14 in the test window.

## Verdict table

The §1 rules applied to §5 (H1, 8 tests) and §6 (H2, 8 tests): Romano–Wolf p < 0.05 and the same sign in the test window.

In [ ]:
verdict = ml.inference.evaluate(reg, {**h1_stats, **h2_stats})
assert len(verdict) == 16
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(verdict.set_index("item")[["statistic", "verdict"]])
print(f"{int((verdict['verdict'] == 'NULL HOLDS').sum())} of 16: NULL HOLDS")

**Reading.** All 16 hypotheses: NULL HOLDS. No overlay changes its base run's Sharpe ratio detectably, and no overlaid core run's Sharpe ratio is detectably different from 60/40's; as in notebook 08, NULL HOLDS is not evidence of equal Sharpe ratios.

ΔSR with its 95% bootstrap interval, H1 and H2 side by side, by base run.

In [ ]:
fp = pd.concat([
    h1.assign(family="H1: VMP(m) − m", label=list(h1["b"])),
    h2.assign(family="H2: VMP(m) − 60/40", label=[a[4:-1] for a in h2["a"]]),
])
fig = ml.plotting.forest_plot(fp, "label", "d_hat_ddof1", "ci_lo", "ci_hi", group="family", ref=0.0,
                              title="ΔSR with bootstrap 95% intervals, full window from 2010-02-02", xlabel="ΔSR")
plt.show()

**Reading.** The H1 intervals straddle 0 for all eight. The H2 intervals are wider for every method and centred below 0; only BL(0.1,S)'s excludes 0.

## §7 Episodes

The two registered episodes, February–April 2020 and calendar 2022 (2020-02-03 to 2020-04-30 and 2022-01-03 to 2022-12-30): cumulative return and in-window maximum drawdown of each base and its overlay, with the overlay's mean and minimum held fraction.

In [ ]:
ep = h.episode_table(all_runs, ORDER)
assert ep.shape[0] == 2 * len(ORDER)
EP_DAYS = {"2020-02/04": ("2020-02-03", "2020-04-30"), "2022": ("2022-01-03", "2022-12-30")}
for e, (a, b) in EP_DAYS.items():
    sel = ep.xs(e, level="episode")
    assert (sel["first"] == pd.Timestamp(a)).all() and (sel["last"] == pd.Timestamp(b)).all()
rows = []
for e in h.EPISODES:
    for m in h.BASES:
        b, v = ep.loc[(m, e)], ep.loc[(h.vmp(m), e)]
        rows.append({"episode": e, "run": m, "ret_base": b["cum_return"], "ret_vmp": v["cum_return"],
                     "dd_base": b["max_dd"], "dd_vmp": v["max_dd"], "mean_h": v["mean_h"], "min_h": v["min_h"]})
epc = pd.DataFrame(rows).set_index(["episode", "run"])
epc_shown = epc.rename(columns={"ret_base": "return, base (%)", "ret_vmp": "return, VMP (%)",
                                "dd_base": "max DD, base (%)", "dd_vmp": "max DD, VMP (%)",
                                "mean_h": "mean h", "min_h": "min h"})
ml.reporting.show_table(epc_shown, pct=["return, base (%)", "return, VMP (%)", "max DD, base (%)", "max DD, VMP (%)"],
                        pct_dp=1)

**Reading.** The overlay cuts crisis drawdowns, but not always crisis losses. In February–April 2020 it cuts the drawdown for all nine runs, to 42% (ERC(S)) to 57% (MDP(S)) of the base, yet cumulative return over the episode is lower for 4 of the nine (GMV(S), MaxSharpe(S), MDP(S) and 60/40). In 2022 the drawdown is shallower for all nine, most for 60/40 (−21.0% to −13.5%), and mean $h$ is 0.58 (60/40) to 0.87 (MDP(S)).

Drawdown within each episode (median across the eight core runs, base solid and VMP dashed; 60/40 and VMP(60/40) in the 60/40 colour) and held fraction $h$.

In [ ]:
fig = h.episode_figure(all_runs, h.CORE)
plt.show()
assert len(fig.axes) == 4

**Reading.** In 2020 the overlay is late: the median core drawdown troughs on 2020-03-19 and the median core $h$ bottoms on 2020-04-07, 13 trading days later, so the overlay is most underweight during the rebound. In 2022 the median core $h$ is below 1 on 89% of trading days.

## §8 Sensitivities

The four registered variants on all nine runs, from 2010-02-02; never tested. Cap 1.5 borrows at BIL with no spread, so it is optimistic.

In [ ]:
sens = h.sensitivity_table(runs, rf_daily, split_ts, ov)
VARS = ["registered"] + list(h.SENS)
sf = sens.xs("full", level="window")


def sens_table(df, caption, dp=2):
    return ml.reporting.show_table(df.loc[h.BASES, VARS].rename_axis(index=caption, columns=None), dp=dp)


for w, lab in (("full", "full"), ("test (> split)", "test")):
    display(sens_table(sens.xs(w, level="window")["dSR_vs_base"].unstack("variant"), f"ΔSR vs base, {lab}"))
display(sens_table(sf["mean_h"].unstack("variant"), "mean h, full"))
sens_table(sf["cost_drag_bps"].unstack("variant"), "cost drag (bp/yr), full", dp=1)

**Reading.** No variant changes the picture. The slower variants trade far less: overlay cost drag is 6–10 bp/yr with a 63-day window and 7–14 with month-end updates, against 19–28 registered. Inverse variance brakes harder (mean $h$ 0.79–0.90); cap 1.5 holds 1.08–1.20 on average at 64–77 bp/yr. Across all 45 variant-run cells the full-window ΔSR vs base stays between −0.15 (MDP(S), inverse variance) and +0.09 (60/40, inverse variance); in the test window it is negative under every variant for 5 runs (EW, BMV(0.3), BL(0.1,S), ERC(S) and 60/40).

Full-window Sharpe of each variant minus the registered overlay's, per run.

In [ ]:
fig = h.sensitivity_heatmap(sens)
plt.show()
assert len(fig.axes) == 2

**Reading.** Against the registered overlay, Sharpe is higher for 6 of 9 runs with the 63-day window and 5 of 9 with month-end updates, most for MDP(S) and GMV(S) in both; it is lower for 8 of 9 with inverse variance and 6 of 9 with cap 1.5.

## §9 TAKEAWAY — the overlay controls drawdowns; its Sharpe effect is too small to detect

- Notebook 09's stress months are where the overlay acts: it cuts the 2020 drawdown to 42–57% of the base for all nine runs.
- The overlay is built on each run's net return, reproduces it exactly at $c = 1$ and uses no future data; it brakes on 28–42% of days at 19–28 bp/yr.
- From 2010-02-02 it lowers volatility (to 73–87% of the base) and return for all nine runs; Sharpe moves by −0.08 (MDP(S)) to +0.09 (60/40).
- H1 and H2: 16 of 16 NULL HOLDS. The overlay does not detectably change any run's Sharpe ratio, and no overlaid core run is detectably different from 60/40; H2 has little power.
- Drawdowns are shallower for all nine in both registered episodes, but in 2020 the overlay is most underweight after the trough.
- Slower variants cut the cost drag to 6–14 bp/yr and narrow the MDP(S) and GMV(S) losses; no variant moves full-window ΔSR vs base outside −0.15 to +0.09.

On these methods the volatility overlay is drawdown control, not a Sharpe improvement: it buys shallower crisis drawdowns with lower return, and its Sharpe effect is too small to detect.